# Layer-Freezing Comparison

This notebook defines an experiment for comparing three RLVR fine-tuning choices: updating all decoder layers, updating only early layers, and updating only later layers. The comparison uses the same dataset, prompt format, and decomposed rewards for every run.

The hypothesis is straightforward: later layers should adapt formatting and task behavior efficiently, early layers may be less efficient because they are closer to lexical representation, and all layers should have the most capacity but also the highest cost and overfitting risk.

In [ ]:
%pip install -q -e ..[train,notebook]

The project runner trains and evaluates the three variants. For a quick smoke test, lower `--max-steps` and the dataset sizes. For a real comparison, keep the seed fixed and increase the number of examples.

In [ ]:
!python ../scripts/run_layer_comparison.py \
  --model Qwen/Qwen3-4B-Instruct-2507 \
  --output-dir ../outputs/layer-comparison \
  --num-train-examples 512 \
  --num-eval-examples 128 \
  --max-steps 100 \
  --layer-fraction 0.33

After the runs finish, load the summary file and compare the reward components. The most useful result is not only the total score: format and count rewards can improve before exact redaction accuracy catches up, while preservation exposes models that mask too aggressively.

In [ ]:
import json
from pathlib import Path

summary_path = Path("../outputs/layer-comparison/summary.json")
reports = json.loads(summary_path.read_text())
rows = []
for report in reports:
    row = {"layer_mode": report["layer_mode"]}
    row.update(report["metrics"])
    rows.append(row)
rows

Pick the model that has the best held-out exact accuracy subject to acceptable preservation. If two runs are close, prefer the cheaper layer setting because it updates fewer parameters and is easier to iterate.